# Projekt: Wskazanie optymalnej lokalizacji farmy fotowoltaicznej, z wykorzystaniem analizy wielokryterialnej 


1. Cel projektu 

Celem projektu było wyznaczenie, bazując na wskazanych kryteriach najlepszej lokalizacji inwestycji farmy fotowoltaicznej. Dodatkowo należało zaprojektować skrypt automatyzujący wybór obszaru najbardziej przydatnego pod wspomnianą wcześniej inwestycję w wybranej gminie. 

2.  Kryteria analizy oraz źródła wykorzystanych danych 

Wybór optymalnej lokalizacji farmy fotowoltaicznej, czyli takiej, która zapewni największą opłacalność i rentowność inwestycji jest procesem skomplikowanym i wymagającym wzięcia pod uwagę wielu czynników. W poniższej tabeli przedstawiono kryteria wykorzystane podczas wykonywania projektu. Jest to jednak tylko wybór przykładowych aspektów, skupiających się głównie wokół danych przestrzennych, spośród wszystkich, które należałoby rozważyć rzeczywiście planując podobną inwestycję. 


# Kryteria oparte na danych zewnętrznych

| Lp. | Kryteria | Parametry | Źródło danych | Opis danych |
|-----|-----------|-----------|---------------|-------------|
| 1   | odległość od rzek i zbiorników wodnych | jak najbliżej; nieprzekraczalna strefa ochronna + bezpieczeństwo powyżej 100m | BDOT10k | Wektorowe warstwy modelujące rzeki i zbiorniki wodne |
| 2   | odległość od budynków mieszkalnych | jak najdalej, powyżej 150m | BDOT10k | Wektorowa warstwa modelująca zabudowę mieszkalną |
| 3   | pokrycie terenu | nie w lesie, powyżej 15m od lasu, optymalnie powyżej 100m od lasu | BDOT10k | Wektorowa warstwa modelująca obszary zalesione |
| 4   | dostęp do dróg utwardzonych | jak największe zagęszczenie | BDOT10k | Warstwa wektorowa modelująca sieć drogową |
| 5   | Nachylenie stoku | optymalnie – płasko; maksymalnie 10% | NMT | Warstwa rastrowa NMT |
| 6   | Dostęp do światła słonecznego | Optymalny: stoki południowe (SW-SE) | NMT | Warstwa rastrowa NMT |
| 7   | Dobry dostęp do głównych dróg i węzłów komunikacyjnych | Najkrótszy czas podróży | OSM | Warstwa wektorowa z punktami przyjętymi jako główne węzły komunikacyjne |

# Kryteria dotyczące otrzymanych w wyniku analizy warstwach

| Lp. | Kryteria | Parametry | Źródło danych | Opis danych |
|-----|-----------|-----------|---------------|-------------|
| 1   | Próg przydatności terenu | 80% / 90% maks. przydatności | - | - |
| 2   | Przydatność działek | Minimum X% działki na obszarze przydatnym | - | - |
| 3   | Minimalna powierzchnia i szerokość obszaru | Powierzchnia 2ha, Szerokość - 50m | - | - |
| 4   | Koszt podłączenia do sieci energetycznej | Najniższy możliwy na analizowanym obszarze | - | - |

3. Założenia wykonanej analizy 

W wyniku analizy wybierany jest obszar składający się z jednej lub więcej działek, który spełnia wszystkie kryteria przy zastosowanych progach przydatności oraz o najniższym koszcie przyłączenia do sieci energetycznej.  

W celu wykonania mapy przydatności, kryteria zostały podzielone ze względu na swoją charakterystykę, na kryteria ostre (podejście z zastosowaniem logiki Boola) i miękkie (podejście z zastosowaniem logiki rozmytej). Kryteria, które zostały przyjęte jako ostre: 
* Odległość obszaru od rzek i zbiorników wodnych musi być większa niż strefa ochronna wynosząca 100 metrów. 
* Odległość obszaru od budynków mieszkalnych musi wynieść przynajmniej 150 metrów. 
* Obszar nie może znajdować się w lesie, a minimalnie 15 metrów od lasu. 

Mapy przydatności powstają zgodnie z metodyką Weighted Linear Combination i z podziałem na metodę wagowania kryteriów miękkich: 
* Wagowanie równe - każde kryterium otrzymuje taką samą wagę 
* Wagowanie różne- każde kryterium otrzymuje wagę przydzielaną uznaniowo 

4. Opis wykonania ćwiczenia 

Analizowanym na potrzeby projektu obszarem była gmina Świeradów-Zdrój. Aby zapewnić jak najlepszą jakość informacji wynikowej, pod uwagę wzięto tereny bezpośrednio sąsiadujące z gminą (w promieniu 150m). 

Analiza poszczególnych kryteriów dla gminy Świeradów-Zdrój: 

1. Odległość od rzek i zbiorników wodnych – jak najbliżej, poza nieprzekraczalną strefą ochronną 100 metrów 

Charakterystyka kryterium wymusza, przy jego analizie, zastosowanie metodyki miękkiej i ostrej. Logika miękka realizuje zależność im bliżej do rzeki/zbiornika wodnego tym lepiej, natomiast logika ostra zapewnia zupełną nieprzydatność terenu znajdującego się w strefie ochronnej 100 metrów od rzek i zbiorników. 

Przebieg analizy przydatności pod kątem kryterium: 

* Zunifikowanie geometrii warstw sieci wodnej do poligonów. 

* Stworzenie obrazu odległości euklidesowych przy użyciu narzędzia Euclidean Distance. Raster powstały wyniku działania tego narzędzia składa się z pikseli którym przypisana jest odległość do najbliższego obiektu warstwy sieci wodnej. 

* Stworzenie obrazu przydatności według metodyki ostrej: 

* Dla warstwy odległości euklidesowych, stworzenie obrazu zgodnie z metodyką rozmytą, stosując funkcję liniową o minimum 0 (metrów) i maksimum 100 (metrów). 

* Reklasyfikacja obrazu, tak, aby wszystkie piksele o wartości innej niż 1 (odległości mniejszej niż 100 metrów) przyjęły wartość 0.    

```python
def getFuzzyMemForSiecWodna() -> None:
    distWoda = "EuCDist_clipped_SiecWodna"
    maxVal = arcpy.management.GetRasterProperties(distWoda, "MAXIMUM").getOutput(0)
    first = arcpy.sa.FuzzyMembership(distWoda, arcpy.sa.FuzzyLinear(0, 100))
    first = arcpy.sa.Reclassify(first, "Value", arcpy.sa.RemapRange([[0, 0.999, 0]]))
    first.save("strong_woda")
    second = arcpy.sa.FuzzyMembership(distWoda, arcpy.sa.FuzzyLinear(maxVal, 102))
    final = arcpy.sa.FuzzyOverlay([first, second], "AND")
    final.save("fuzzy_siec_wodna")
```


2. Odległość od budynków mieszkalnych – jak najdalej, powyżej 150 metrów 

To kryterium również wymaga analizy zgodnie z metodykami rozmytą i ostrą. W tym przypadku logika rozmyta realizuje założenie: im dalej od budynków mieszkalnych tym przydatność jest większa, natomiast logika ostra zapewnia, że obszary w odległości do 150 metrów od budynków mieszkalnych są uznawane za nieprzydatne. 

Przebieg analizy przydatności pod kątem kryterium: 

* Wyselekcjonowanie z warstwy wektorowej budynków (BUBD_A z bazy danych BDOT10k) warstwy budowli o przeznaczeniu mieszkalnym. 

* Dla nowopowstałej warstwy stworzenie warstwy odległości euklidesowych. 

* Stworzenie obrazu przydatności zgodnego z metodyką rozmytą, który otrzymujemy poprzez zastosowanie funkcji FuzzyMembership z funkcją liniową o minimum wynoszącym 150 metrów i maksimum wynoszącym maksymalną odległość pobraną z rastra odległości euklidesowych 

* Obraz przydatności dla metodyki ostrej otrzymujemy reklasyfikując obraz metodyki miękkiej, nadpisując wartość 1 wszystkim pikselom, które nie są zerami (czyli są oddalone od budynków mieszkalnych o przynajmniej 152 metry): 


3. Pokrycie terenu – odległość od lasu powyżej 15 metrów, optymalnie powyżej 100 metrów 

Kryterium swoją treścią wymusza zastosowanie do jego realizacji metodyki ostrej i rozmytej. Kryterium ostrym w tym przypadku jest odległość od lasu wynosząca przynajmniej 15 metrów, stosując logikę ostrą dla tego fragmentu kryterium minimalizujemy wpływ cieni rzucanych przez drzewa na wydajność inwestycji. Kryterium miękkie gwarantuje liniowo rosnącą przydatność w przedziale 15- 100 metrów i przydatność maksymalną powyżej 100 metrów. 

Przebieg analizy przydatności pod kątem kryterium: 

* Stworzenie warstwy odległości euklidesowych dla warstwy wektorowej lasów. 

* Wygenerowanie obrazu przydatności logiką ostrą: 

* Stworzenie przy użyciu narzędzia FuzzyMembership obraz przydatności zgodny z metodyką miękką, stosując funkcję liniową o minimum równym 0 metrów i maksimum równym 15 metrów. 

* Reklasyfikacja uprzednio powstałej warstwy, nadpisując wszystkim wartościom innym niż 1 wartości 0 – wszystkie obszary w odległości do 15 metrów od lasów otrzymują zerową przydatność. 

* Wygenerowanie obrazu przydatności logiką rozmytą używając narzędzia FuzzyMembership z funkcją liniową o minimum równym 17 metrów i maksimum równym 100 metrów. 


4. Dostęp do dróg utwardzonych – jak największe zagęszczenie 

Przy analizie tego kryterium wykorzystać należało funkcję LineDensity na warstwie dróg, obliczającą gęstość obiektów w przeliczeniu na jednostkę powierzchni (przyjęto kilometry kw.). Następnie na warstwie wynikowej funkcji LineDensity zastosowana została funkcja FuzzyMembership z funkcją liniową o minimum równym 0 i maksimum równym wartości maksymalnej pobranej z warstwy gęstości dróg. 


5. Nachylenie stoków - im mniejsze tym lepiej, do 10% 

Przebieg analizy przydatności pod kątem kryterium: 

* Wygenerowanie mapy spadków na podstawie numerycznego modelu terenu. 

* Stworzenie obrazu przydatności zgodnie z metodyką rozmytą (funkcja FuzzyMembership) stosując funkcję liniową. Za minimum funkcja przyjmuje arbitralnie przyjętą największą wartość kątową, dla której dany piksel jest idealnie przydatny (przyjęto 5% nachylenia), a za maksimum 0 (% nachylenia). 

* Reklasyfikacja uprzednio otrzymanego obrazu, nadając wszystkim pikselom o wartości 0%-5% wartości (przydatności) 1. (Obraz nr 1) 

* Stworzenie kolejnego obrazu przydatności stosując funkcję FuzzyMembership na podstawie mapy spadków. Ponownie użycie funkcji liniowej, tym razem za minimum przyjmując 10 (% nachylenia), a za maksimum 0 (% nachylenia). W ten sposób otrzymujemy obraz, w którym wszystkie piksele o nachyleniu większym niż 10 % otrzymują wartość 0, natomiast przydatność rośnie od 10% w kierunku 0%. (Obraz nr 2) 

* Wygenerowanie ostatecznej mapy przydatności dla tego kryterium poprzez połączenie logiką “OR” obrazu nr 1 i obrazu nr 2 (funkcja FuzzyOverlay). 

6. Dostęp do światła słonecznego - optymalnie stoki południowe 

Przebieg analizy przydatności pod kątem kryterium: 

* Wygenerowanie mapy orientacji nachylenia stoków - funkcja Aspect. 

* Reklasyfikacja otrzymanej mapy nadając wartość 1 pikselom o wartości znajdującej się w przedziałach kątów azymutalnych odpowiadających stokom południowym, południowo-zachodnich, południowo-wschodnich: 
    * ⟨−1⟩
    * ⟨112.5, 247.5⟩
    I wartości 0 pikselom o wartościach z przedziałów: 
    * ⟨0, 112.5⟩
    * ⟨247.5, 360⟩


W przypadku tego kryterium do wykonania mapy przydatności wykorzystano metodykę ostrą, pomimo faktu, iż w na etapie tworzenia zbiorczej mapy przydatności, nie jest traktowane jako kryterium ostre. Takie podejście uzasadnione jest tym, iż kierunek nachylenia stoku jest czynnikiem, nie koniecznym, ale bardzo istotnym z perspektywy rentowności inwestycji jaką jest farma fotowoltaiczna. Farma zlokalizowana na stoku o optymalnej orientacji może zapewniać wydajność energetyczną wyższą o nawet 30%*, stąd choć kryterium traktowane jest jako miękkie, poprzez nadanie przydatności 1 lub zero, będzie miało duży wpływ na ostateczną (zbiorczą) przydatność na terenie wybranej gminy. 



7. Jak najkrótszy dojazd do istotnych drogowych węzłów komunikacyjnych 

Przebieg analizy przydatności pod kątem kryterium: 

* Wybór punktów odpowiadających istotnym drogowym węzłom komunikacyjnym. Dla gminy Świeradów-Zdrój wybranych zostało 16 punktów, zlokalizowanych na odcinku Jelenia Góra - Lubań drogi krajowej nr 30. Są to punkty, w których występuje skrzyżowanie wspomnianej drogi z drogami, którymi można dojechać do drogi krajowej nr 30 ze Świeradowa-Zdroju. 

* Wygenerowanie obrazu czasu dojazdu do węzłów komunikacyjnych. Czynność ta wykonywana jest przy użyciu wtyczki programu QNEAT3 QGIS, która oferuje funkcje przydatne przy analizach sieciowych. Wynikowy obraz składa się z pikseli, których wartość zależna jest od czasu dojazdu do punktów wybranych jako dane wejściowe dla wtyczki – w naszym przypadku punktów reprezentujących istotne węzły komunikacyjne. Ważne jest odpowiednie dopasowanie zasięgu obliczeń w menu narzędzia QNEAT, tak aby ogarniał on jak największą część obszaru opracowania (tu obszaru gminy). 

* Pobranie maksymalnej i minimalnej wartości z rastra czasu dojazdu do węzłów komunikacyjnych. 

* Przypisanie pikselom o wartościach 0 (No Data), wartości maksymalnej pobranej w poprzednim punkcie. 

* Wygenerowanie obrazu przydatności dla kryterium, stosując metodykę rozmytą z funkcją liniową o minimum równemu wartości maksymalnej pobranej z rastra oraz maksimum równemu wartości minimalnej. 

 

Połączenie wyżej wymienionych kryteriów w zbiorczą mapę przydatności dla gminy Świeradów-Zdrój: 

Specyfika projektu zakładała finalne wygenerowanie dwóch map przydatności: stosującej równe wagi kryteriów miękkich i stosującej różne nadane arbitralnie wagi kryteriów miękkich. 
W drugim przypadku zastosowano następujące wagi: 
| Kryterium                               | Waga |
|-----------------------------------------|------|
| Odległość od rzek i zbiorników wodnych  | 0.10 |
| Odległość od budynków mieszkalnych      | 0.15 |
| Pokrycie Terenu                         | 0.20 |
| Dostęp do dróg utwardzonych             | 0.20 |
| Nachylenie stoków                       | 0.10 |
| Dostęp do światła słonecznego           | 0.20 |
| Dojazd do istotnych węzłów komunikacyjnych | 0.05 |

Kryteria miękkie zostały połączone z ostrymi zgodnie z wzorem na Weighted Linear Combination: 
$$P = \sum w_i X_i \cdot \prod C_j$$ 
Gdzie: 
<br>P - przydatność całkowita <br>
$X_{i}$ - kryterium miękkie <br>
$C_{j}$ - kryteria ostre wyrażone w jednym obrazie <br>
$\prod$ - symbol iloczynu

W środowisku arcgis do wygenerowanie nieznormalizowanych map przydatności użyto kombinacji funkcji Weighted Sum i Times.  

Ostateczne mapy przydatności otrzymywane są poprzez przyjęcie progu przydatności równemu 70% maksymalnej przydatności nieznormalizowanej i nadanie odpowiednio wartości 0 pikselom mającym wartość poniżej progu i 1 pikselom powyżej progu.

